# Conductor data, multi-source (CPU session, no GPU quota)
Settings: Accelerator **None**, Internet **ON**. Output: `conductor_rows.jsonl`, `conductor_rows_summary.json`.
Pulls specific parquet files from four datasets, converts each with its own adapter (`tools/conductor_data.py adapt`), applies ONE label rule, and splits by repo across ALL sources together:
nebius/SWE-rebench-openhands-trajectories (CC-BY-4.0), SWE-bench/SWE-smith-trajectories `tool` shards (MIT), nanoswe/nanoswe-trajs-261002 spread shards (MIT), JetBrains-Research/agent-trajectories-swesmith-random-subset (MIT, held out as a different-agent test).
After it finishes: **Save Version** (Save & Run All), then attach this notebook's output to the GPU notebook as an input.

In [ ]:
import subprocess, sys, shutil
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyarrow", "huggingface_hub"], check=True)
print("free GB on /kaggle/working:", shutil.disk_usage("/kaggle/working").free / 1e9, "| /kaggle/temp:", shutil.disk_usage("/kaggle/temp").free / 1e9)

In [ ]:
open('/kaggle/working/conductor_data.py', 'w', encoding='utf-8').write('"""Builds Laya-conductor fine-tune rows (`should_continue`) from nebius/SWE-rebench-openhands-trajectories (PLAN.md 13.7).\n\nRow format = the official conductor notebook\'s: {"state": {...}, "gold": {"should_continue": {"probabilities": {"false": p, "true": q}}}, ...meta}.\n`should_continue` = P(agent stopped mid-task with work left) = the plan\'s `done` label negated: FALSE only at the final step of a resolved\ntrajectory; TRUE at every earlier step and at the final "finish" of an unresolved one.\n\nWhy last_observation: at the final step the response ("the fix is complete...") reads alike whether the run later resolved or not, so text alone cannot\nseparate a resolved finish from a premature one. The tool result just before the step (observable at t, never the step itself or later) is the evidence.\n\nlaya.common.build_sequence serializes the state with json.dumps and keeps the HEAD of it (truncate_left=False), so a long field silently\npushes later fields out. Hence: last_response first, all fields cut to fixed char budgets (tuned by tools/check_conductor_rows.py).\n"""\nimport random\n\nTASK_CHARS = 600        # head of the first user message (the issue)\nRESP_CHARS = 1000       # TAIL of the assistant message (its conclusion / last action)\nOBS_CHARS = 900         # TAIL of the last tool result before this step (test output etc.): the only execution evidence the state carries\nK_MID = 2               # random intermediate steps per trajectory, plus the last and the one before it\n\n\ndef _text(m):\n    t = m.get("content") or ""\n    for c in m.get("tool_calls") or []:\n        f = c.get("function", {})\n        t += f"\\n[tool {f.get(\'name\')}] {f.get(\'arguments\') or \'\'}"\n    return t\n\n\ndef _unwrap(c):\n    """SWE-smith stores observations as the string repr of [{\'type\': \'text\', \'text\': ...}]; return the text."""\n    if isinstance(c, list):\n        return "".join(x.get("text", "") for x in c if isinstance(x, dict))\n    if isinstance(c, str) and c.startswith("[{\'type\'"):\n        import ast\n        try:\n            return "".join(x.get("text", "") for x in ast.literal_eval(c))\n        except Exception:\n            return c\n    return c or ""\n\n\ndef _repo_of(instance_id):\n    """SWE-smith ids look like owner__repo.<hash>.<bug>; the repo is what the split must key on."""\n    return instance_id.split(".")[0].replace("__", "/", 1)\n\n\nSOURCES = {   # source -> (outcome column, repo column or None)\n    "nebius": ("resolved", "repo"),\n    "swesmith_tool": ("resolved", None),\n    "jetbrains_mini": ("resolved", None),\n    "nanoswe": ("verified", None),\n}\n\n\ndef adapt(rec, source):\n    """One raw dataset row -> the common form make_rows reads: {trajectory, resolved, trajectory_id, repo, source}.\n    Common form: roles user (first = the task), assistant (one per step, tool calls as `tool_calls` or an `action` string), tool (what came back)."""\n    import json as _j\n    outcome, repo_col = SOURCES[source]\n    if source == "nebius":\n        return {**rec, "source": source}\n    msgs = rec["messages"]\n    if isinstance(msgs, str):\n        msgs = _j.loads(msgs)\n    if source == "swesmith_tool":\n        out = [{"role": m["role"], "content": _unwrap(m.get("content")) if m["role"] != "assistant" else (m.get("content") or ""),\n                "tool_calls": [{"function": {"name": "action", "arguments": m["action"]}}] if m["role"] == "assistant" and m.get("action") else None}\n               for m in msgs if m["role"] != "system"]\n    elif source == "jetbrains_mini":    # mini-swe-agent: assistant THOUGHT + bash block, command output comes back as a user message\n        seen_user, out = False, []\n        for m in msgs:\n            if m["role"] == "system":\n                continue\n            role = m["role"]\n            if role == "user":\n                role, seen_user = ("user" if not seen_user else "tool"), True\n            out.append({"role": role, "content": m["content"], "tool_calls": None})\n    else:                                # nanoswe: OpenAI-style, unverified beyond role=user first; unknown roles fail loudly in make_rows\' caller\n        out = [{"role": m["role"], "content": m.get("content") or "", "tool_calls": m.get("tool_calls")} for m in msgs if m["role"] != "system"]\n        assert {m["role"] for m in out} <= {"user", "assistant", "tool"}, {m["role"] for m in out}\n    iid = rec["instance_id"]\n    return {"trajectory": out, "resolved": bool(rec[outcome]), "trajectory_id": rec.get("traj_id") or rec.get("traj_hash") or f"{source}:{iid}",\n            "repo": rec[repo_col] if repo_col else _repo_of(iid), "instance_id": iid, "source": source}\n\n\ndef make_rows(rec, rng, split):\n    """rec: one dataset row (trajectory list of dicts). Returns a list of training rows for this trajectory."""\n    msgs = rec["trajectory"]\n    task = next((m.get("content") or "" for m in msgs if m["role"] == "user"), "")[:TASK_CHARS]\n    steps = [i for i, m in enumerate(msgs) if m["role"] == "assistant"]\n    if len(steps) < 3:\n        return []\n    last = len(steps) - 1\n    pick = {last, last - 1} | set(rng.sample(range(0, last - 1), min(K_MID, last - 1)))\n    resolved = int(rec["resolved"])\n    rows = []\n    for s in sorted(pick):\n        stopped_with_work_left = not (s == last and resolved == 1)\n        p_true = 1.0 if stopped_with_work_left else 0.0\n        obs = next((m.get("content") or "" for m in reversed(msgs[:steps[s]]) if m["role"] == "tool"), "")\n        rows.append({\n            "state": {"last_response": _text(msgs[steps[s]])[-RESP_CHARS:], "last_observation": obs[-OBS_CHARS:], "task": task},\n            "gold": {"should_continue": {"probabilities": {"false": 1.0 - p_true, "true": p_true}}},\n            "trajectory_id": rec["trajectory_id"], "repo": rec["repo"], "source": rec.get("source", "nebius"), "split": split,\n            "step": s, "n_steps": len(steps), "is_last": s == last, "resolved": resolved,\n            # plan targets kept separate (PLAN.md 13.3): the conductor trains on should_continue only; `done` and `value` are stored for the internal head and are never merged\n            "done": int(s == last and resolved == 1), "value": resolved,\n        })\n    return rows\n\n\ndef demo():\n    rng = random.Random(0)\n    msg = lambda r, c: {"role": r, "content": c}\n    traj = [msg("system", "s"), msg("user", "fix bug")] + [msg("assistant", f"step {i}") for i in range(6)]\n    for resolved in (1, 0):\n        rows = make_rows({"trajectory": traj, "resolved": resolved, "trajectory_id": "t", "repo": "r"}, rng, "train")\n        by_last = {r["step"]: r["gold"]["should_continue"]["probabilities"]["true"] for r in rows}\n        assert by_last[5] == (0.0 if resolved else 1.0), by_last          # final step: FALSE only when resolved\n        assert all(v == 1.0 for s, v in by_last.items() if s != 5), by_last  # every earlier step: TRUE\n        assert {4, 5} <= set(by_last)\n    assert all("last_observation" in r["state"] for r in rows)\n    t2 = traj[:3] + [{"role": "tool", "content": "3 passed"}] + traj[3:]\n    r2 = make_rows({"trajectory": t2, "resolved": 1, "trajectory_id": "t", "repo": "r"}, rng, "train")\n    assert any(r["state"]["last_observation"] == "3 passed" for r in r2) and r2[0]["state"]["last_observation"] in ("", "3 passed")\n    assert make_rows({"trajectory": traj[:4], "resolved": 1, "trajectory_id": "t", "repo": "r"}, rng, "train") == []\n    sm = {"instance_id": "owner__repo.abc.bug_1", "resolved": True, "traj_id": "x", "messages": [{"role": "system", "content": "s"},\n          {"role": "user", "content": "[{\'type\': \'text\', \'text\': \'fix it\'}]"}] + sum([[{"role": "assistant", "content": f"t{i}", "action": f"ls {i}"},\n          {"role": "tool", "content": "[{\'type\': \'text\', \'text\': \'out" + str(i) + "\'}]"}] for i in range(4)], [])}\n    a = adapt(sm, "swesmith_tool")\n    assert a["repo"] == "owner/repo" and a["trajectory"][0]["content"] == "fix it" and a["trajectory"][2]["content"] == "out0"\n    r = make_rows(a, rng, "train"); assert r and r[0]["source"] == "swesmith_tool" and r[-1]["state"]["last_observation"].startswith("out")\n    jb = {"instance_id": "o__r.h.b", "resolved": False, "messages": [{"role": "system", "content": "s"}, {"role": "user", "content": "task"}] +\n          sum([[{"role": "assistant", "content": f"THOUGHT {i}"}, {"role": "user", "content": f"<returncode>0</returncode>{i}"}] for i in range(4)], [])}\n    a = adapt(jb, "jetbrains_mini"); assert [m["role"] for m in a["trajectory"][:4]] == ["user", "assistant", "tool", "assistant"], a["trajectory"][:4]\n    assert make_rows(a, rng, "train")[-1]["gold"]["should_continue"]["probabilities"]["true"] == 1.0     # unresolved: final is TRUE\n    print("conductor_data demo ok")\n\n\nif __name__ == "__main__":\n    demo()\n')
exec(open('/kaggle/working/conductor_data.py', encoding='utf-8').read().replace('if __name__', 'if False and __name__'))

In [ ]:
import json, hashlib, random, collections
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download

HELD_OUT = "jetbrains_mini"   # a different agent and message format: never in train, and its repos are removed from train (source-held-out test)
# name: (hf dataset, files, id column, outcome column, exit_status column or None, trajectories wanted, per-repo cap, message column)
SRC = {
 "nebius":         ("nebius/SWE-rebench-openhands-trajectories", ["trajectories.parquet"], "trajectory_id", "resolved", "exit_status", 2000, 12, "trajectory"),
 "swesmith_tool":  ("SWE-bench/SWE-smith-trajectories", [f"data/tool-0000{i}-of-00008.parquet" for i in (0, 3, 6)], "traj_id", "resolved", None, 1000, 8, "messages"),
 "nanoswe":        ("nanoswe/nanoswe-trajs-261002", [f"train-{i:05d}.parquet" for i in (0, 25, 50, 75, 100, 125, 150)], "traj_hash", "verified", "exit_status", 1000, 8, "messages"),
 "jetbrains_mini": ("JetBrains-Research/agent-trajectories-swesmith-random-subset", ["data/train-00000-of-00001.parquet"], None, "resolved", "exit_status", 500, 8, "messages"),
}
SUBMIT = {"submit", "Submitted"}      # exit_status values meaning the agent chose to finish
TEST_FRAC = 0.2                       # repo-disjoint test split over the non-held-out sources

def is_test(repo): return int(hashlib.sha256(repo.encode()).hexdigest(), 16) % 100 < TEST_FRAC * 100

paths = {n: [hf_hub_download(s[0], f, repo_type="dataset") for f in s[1]] for n, s in SRC.items()}
print({n: len(v) for n, v in paths.items()}, "files downloaded; free GB on /kaggle/working:", shutil.disk_usage("/kaggle/working").free / 1e9)

def keyof(name, idc, rec, path, rg, i):
    return str(rec.get(idc) or f"{name}:{path[-14:]}:{rg}:{i}")

def repo_of(rec):
    return rec.get("repo") or rec["instance_id"].split(".")[0].replace("__", "/", 1)

sel, report = {}, {}
rngp = random.Random(1)
for name, (ds, files, idc, oc, ec, want, cap, mc) in SRC.items():
    ms = []
    for path in paths[name]:
        f = pq.ParquetFile(path)
        have = [c for c in (idc, oc, ec, "instance_id", "repo") if c and c in f.schema_arrow.names]
        for rg in range(f.num_row_groups):
            for i, m in enumerate(f.read_row_group(rg, columns=have).to_pylist()):
                m["_key"] = keyof(name, idc, m, path, rg, i); m["_repo"] = repo_of(m); ms.append(m)
    elig = [m for m in ms if ec is None or m.get(ec) in SUBMIT]
    rngp.shuffle(elig)
    per, pick = collections.Counter(), {0: [], 1: []}
    for m in elig:
        r = int(bool(m[oc]))
        if per[m["_repo"]] >= cap or len(pick[r]) >= want // 2: continue
        per[m["_repo"]] += 1; pick[r].append(m)
    sel[name] = {m["_key"]: m for m in pick[0] + pick[1]}
    report[name] = dict(scanned=len(ms), eligible=len(elig), selected=len(sel[name]), resolved=len(pick[1]), repos=len({m["_repo"] for m in sel[name].values()}))
    print(name, report[name])

held_repos = {m["_repo"] for m in sel[HELD_OUT].values()}
for name in sel:
    if name != HELD_OUT:       # leak guard: any repo that appears in the held-out source leaves the train/test pool
        sel[name] = {k: m for k, m in sel[name].items() if m["_repo"] not in held_repos}
print("held-out repos:", len(held_repos), "| selected after removal:", {n: len(v) for n, v in sel.items()})

rng = random.Random(2)
rows = []
for name, (ds, files, idc, oc, ec, want, cap, mc) in SRC.items():
    for path in paths[name]:
        f = pq.ParquetFile(path)
        have = [c for c in (idc, oc, "instance_id", "repo", mc) if c and c in f.schema_arrow.names]
        for rg in range(f.num_row_groups):
            for i, rec in enumerate(f.read_row_group(rg, columns=have).to_pylist()):
                key = keyof(name, idc, rec, path, rg, i)
                if key not in sel[name]: continue
                rec.setdefault("instance_id", key)
                if name == "nebius":
                    a = {**rec, "source": name}
                else:
                    rec["messages"] = rec[mc]; a = adapt(rec, name)
                a["trajectory_id"] = key
                split = "test_ood" if name == HELD_OUT else ("test" if is_test(a["repo"]) else "train")
                rows += make_rows(a, rng, split)
with open("/kaggle/working/conductor_rows.jsonl", "w", encoding="utf-8") as out:
    for r in rows: out.write(json.dumps(r, ensure_ascii=False) + "\n")
by = lambda s: [r for r in rows if r["split"] == s]
share = lambda xs: sum(r["gold"]["should_continue"]["probabilities"]["true"] for r in xs) / max(len(xs), 1)
summary = dict(sources=report, rows=len(rows), splits={s: dict(rows=len(by(s)), true_share=share(by(s)), final_rows=sum(r["is_last"] for r in by(s)), repos=len({r["repo"] for r in by(s)})) for s in ("train", "test", "test_ood")},
               rows_per_source_split={f"{n}/{s}": sum(1 for r in rows if r["source"] == n and r["split"] == s) for n in SRC for s in ("train", "test", "test_ood")},
               repo_overlap_train_test=len({r["repo"] for r in by("train")} & {r["repo"] for r in by("test")}),
               repo_overlap_train_ood=len({r["repo"] for r in by("train")} & {r["repo"] for r in by("test_ood")}),
               params=dict(HELD_OUT=HELD_OUT, TEST_FRAC=TEST_FRAC, SRC={n: dict(files=s[1], want=s[5], cap=s[6]) for n, s in SRC.items()}, TASK_CHARS=TASK_CHARS, RESP_CHARS=RESP_CHARS, OBS_CHARS=OBS_CHARS, K_MID=K_MID))
assert summary["repo_overlap_train_test"] == 0 and summary["repo_overlap_train_ood"] == 0
json.dump(summary, open("/kaggle/working/conductor_rows_summary.json", "w"), indent=1)
print(json.dumps(summary, indent=1))